# 06 — Test metrics & change maps

Loads the best checkpoint from Drive, computes final test metrics, and renders per-city change maps (t0 / t1 RGB, ground truth, predicted probability) for the report. Figures are pushed to Drive.

**Storage contract** — code lives on GitHub, data & checkpoints live on Drive (`MyDrive/s2cd`), `/content` is disposable scratch. Every notebook is self-contained: run all cells top-to-bottom in a fresh GPU session.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import os

REPO_URL = "https://github.com/Aadith-Geeth-Mohan/sentinel2-change-detection.git"
REPO = "/content/s2-change-detection"
if os.path.exists(REPO):
    !git -C {REPO} pull --ff-only
else:
    !git clone {REPO_URL} {REPO}
%cd {REPO}
!pip install -q .
import sys
sys.path.insert(0, os.path.join(REPO, "src"))
# drop cached s2cd modules so a just-pulled fix takes effect in THIS kernel
for k in [k for k in list(sys.modules) if k == "s2cd" or k.startswith("s2cd.")]:
    del sys.modules[k]


In [ ]:
import torch
from s2cd.data import DriveStore
from s2cd.utils import get_device, describe_device, load_config

device = get_device()
print(describe_device(device))
store = DriveStore()  # Drive root: MyDrive/s2cd · scratch: /content


In [ ]:
from s2cd.evaluate import load_best_model, test_metrics

model, cfg, device = load_best_model("configs/train.yaml", store)
manifest = store.read_manifest()
results = test_metrics(model, cfg, store, manifest)
import json
print(json.dumps(results, indent=1))


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from s2cd.data.dataset import load_stats_from_manifest
from s2cd.data.oscd import list_cities, read_city
from s2cd.evaluate import predict_city

stats = load_stats_from_manifest(manifest)
bands = load_config("configs/data.yaml")["bands"]
rgb = [bands.index("B04"), bands.index("B03"), bands.index("B02")]
oscd_root = store.extract_oscd()

for city_name in sorted(list_cities(oscd_root)["test"])[:3]:
    c = read_city(oscd_root, city_name, bands)
    prob = predict_city(model, c["t0"], c["t1"], stats, chip=256, stride=256)

    fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
    stretch = lambda a: np.clip(a[[rgb]] / 0.3, 0, 1).transpose(1, 2, 0)
    axes[0].imshow(stretch(c["t0"])); axes[0].set_title(f"{city_name} t0 ({c['dates'][0]})")
    axes[1].imshow(stretch(c["t1"])); axes[1].set_title(f"{city_name} t1 ({c['dates'][1]})")
    axes[2].imshow(c["mask"], cmap="gray"); axes[2].set_title("ground truth")
    axes[3].imshow(prob, cmap="magma", vmin=0, vmax=1); axes[3].set_title("P(change)")
    for ax in axes:
        ax.set_axis_off()
    plt.tight_layout()
    rep = store.scratch / "reports"; rep.mkdir(exist_ok=True)
    fig.savefig(rep / f"map_{city_name}.png", dpi=150, bbox_inches="tight")
    store.push_report(f"map_{city_name}.png")
    plt.show()


In [ ]:
import json
rep = store.scratch / "reports"
(rep / "test_metrics.json").write_text(json.dumps(
    {**results, "device": describe_device(device)}, indent=1))
store.push_report("test_metrics.json")
print("pushed test_metrics.json -> Drive")
